# 08 · Reinforcement learning on code

RLOO with executable rewards (`glm_ladder/rl.py`) on the two joint fine-tuned models. The result of this notebook is the project's output: one RL-trained model per size.

**Resumable.** Every result goes to the runs repo as soon as it exists: the baselines, the filtered training sets, an RL checkpoint (weights, AdamW state, RNG, history) every 25 steps, and each run's final measurements. If the session dies, open a new runtime and run all cells again: finished work is downloaded instead of recomputed, and an unfinished run continues from its last checkpoint (at most 25 steps are lost).

| Section | What | A100 time (estimate) |
|---|---|---:|
| 6 | Models, tasks, data | 3 min |
| 7–8 | Baseline measurements of both starting models | 10 min |
| 9 | Difficulty filter (32 answers per training task) | 4 min |
| 10 | Three runs: `s-50m`, `m-100m`, `s-50m` without KL (200 steps, 16 tasks x 16 answers) | ~80 min |
| 11 | Before vs after, paired by task | 1 min |

**Evaluation.** Generated held-out tasks (`test_id`: new parameters of trained families, `test_ood`: families never trained on) and the 40 hand-written tasks ("near transfer"). 64 samples per task at temperature 1.0 (the distribution RL trains) plus one greedy answer. Before/after differences are paired by task, with a bootstrap 95% interval. Also measured before and after: review rating (test sets), SFT validation loss, code reference loss, Turkish format leakage, chat samples.

**Safety.** Generated code runs in a subprocess with a deny-list, a timeout, a read-only temporary directory, an empty environment, limits on memory, CPU time and file size, and as the unprivileged user `nobody`. The token stays in this kernel's memory during RL (checkpoints need it), but a `nobody` process cannot read the memory or files of the root process that holds it.

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100***.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
# In VS Code + Colab, "Restart Kernel" can leave the old kernel (and its compile workers)
# running and holding GPU memory, which later causes CUDA out-of-memory errors.
import os
import signal

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid == me:
        continue
    if "colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args):
        stale.append((pid, args[:80]))
for pid, args in stale:
    os.kill(pid, signal.SIGKILL)
    print("killed stale process", pid, args)
print("stale processes:", len(stale))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Run all tests

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 6. Models, tasks, data, and the receipt store

`publish(name, data)` writes a JSON receipt locally and uploads it at once to `rl/` in the runs repo; `fetch(name)` downloads it if it is there. Every expensive result below goes through these two, which is what makes the notebook resumable.

In [ ]:
import copy
import json
import shutil
import time

import numpy as np
import torch
from huggingface_hub import snapshot_download

sys.path.insert(0, str(REPO_DIR / "scripts"))
from glm_ladder.checkpoint import sha256_file
from glm_ladder.code_eval import load_tasks
from glm_ladder.runs import RunStore
from glm_ladder.sft_data import SFTSource
from glm_ladder.tokenizer import load_tokenizer
from sample import load_run                    # scripts/sample.py: downloads, checks SHA-256

OUT = Path("/content/rl-receipts")
OUT.mkdir(exist_ok=True)


def publish(name, data):
    path = OUT / f"{name}.json"
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    api.upload_file(path_or_fileobj=str(path), path_in_repo=f"rl/{name}.json", repo_id=RUNS_REPO,
                    repo_type="model", commit_message=f"rl: {name}")


def fetch(name):
    if not api.file_exists(RUNS_REPO, f"rl/{name}.json", repo_type="model"):
        return None
    path = api.hf_hub_download(repo_id=RUNS_REPO, filename=f"rl/{name}.json", repo_type="model", force_download=True)
    return json.loads(Path(path).read_text(encoding="utf-8"))


STARTS = {"s50m": "mix-s50m-lr1e-04", "m100m": "mix-m100m-lr1e-04"}
base = {}
for size, run in STARTS.items():
    model, state = load_run(api, RUNS_REPO, run, None)
    base[size] = (model, state)
    print(f"{size}: {run} @ {state['checkpoint']}")

DATA_DIR = Path("/content/data")
snapshot_download(repo_id=DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=token,
                  allow_patterns=["finetune/manifest.json", "finetune/*/test.jsonl", "sft/manifest.json", "sft/*/val/*"])
ft_manifest = json.loads((DATA_DIR / "finetune" / "manifest.json").read_text())
sft_manifest = json.loads((DATA_DIR / "sft" / "manifest.json").read_text())
for lang in ("tr", "en"):
    assert sha256_file(DATA_DIR / "finetune" / lang / "test.jsonl") == ft_manifest["languages"][lang]["test"]["sha256"]["test.jsonl"]
for name, splits in sft_manifest["sources"].items():
    for file, digest in splits["val"]["sha256"].items():
        assert sha256_file(DATA_DIR / "sft" / name / "val" / file) == digest
print("data verified")

tokenizer = load_tokenizer(REPO_DIR / "tokenizer" / "glm-ladder-16k.json")
TASKS = {split: load_tasks(REPO_DIR / "evals" / "rl_tasks" / f"{split}.jsonl") for split in ("train", "test_id", "test_ood")}
EVAL_SETS = {"test_id": TASKS["test_id"], "test_ood": TASKS["test_ood"],
             "hand_written": load_tasks(REPO_DIR / "evals" / "code_tasks.jsonl")}
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")
GPU = torch.cuda.get_device_name()
print({k: len(v) for k, v in {**TASKS, **EVAL_SETS}.items()})

## 7. Measurements

One function measures everything for a model, so "before" and "after" are measured identically.

In [ ]:
from glm_ladder.chat import END_TOKEN, prompt_ids, special_id
from glm_ladder.classify import LANGUAGES, evaluate_reviews, label_token_ids, score
from glm_ladder.evaluation import evaluate as loss_evaluate
from glm_ladder.rl import Sampler, pass_counts, summarize_counts
from run_code_eval import evaluate as code_evaluate

N_SAMPLES = 64
EVAL_TEMPERATURE = 1.0


def read_jsonl(path):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    return [row["text"] for row in rows], [row["stars"] for row in rows]


reviews = {lang: read_jsonl(DATA_DIR / "finetune" / lang / "test.jsonl") for lang in LANGUAGES}
sft_val = {f"sft_{name}": SFTSource(DATA_DIR / "sft" / name / "val") for name in sft_manifest["sources"]}
GENERAL = {
    "tr": ["Sağlıklı beslenme için üç öneri ver.", "İstanbul hakkında kısa bilgi verir misin?",
           "Bir e-posta için kibar bir kapanış cümlesi yaz.", "Fotosentez nedir? İki cümleyle açıkla.",
           "Python'da liste ile demet arasındaki fark nedir?", "Bana kısa bir hikaye anlat.",
           "Sabah rutini için beş öneri yaz.", "Türkiye'nin başkenti neresidir?",
           "İyi bir özgeçmiş nasıl yazılır?", "Bu cümleyi daha resmi yap: yarın gelemiyorum.",
           "Kahve ile çay arasındaki farklar neler?", "Spor yapmaya nasıl başlarım?",
           "Bir toplantı gündemi örneği hazırla.", "Yağmur hakkında kısa bir şiir yaz.",
           "Ders çalışırken nasıl odaklanırım?", "Bana bir kitap önerir misin?",
           "Su döngüsünü açıkla.", "Arkadaşıma doğum günü mesajı yaz.",
           "Bütçe yapmanın faydaları nelerdir?", "Bilgisayarım yavaşladı, ne yapabilirim?"],
    "en": ["Give me three tips for healthy eating.", "Tell me a little about London.",
           "Write a polite closing line for an email.", "What is photosynthesis? Explain in two sentences.",
           "What is the difference between a list and a tuple in Python?", "Tell me a short story.",
           "Write five tips for a morning routine.", "What is the capital of France?",
           "How do I write a good resume?", "Make this sentence more formal: can't come tomorrow.",
           "What are the differences between coffee and tea?", "How do I start exercising?",
           "Draft a sample meeting agenda.", "Write a short poem about rain.",
           "How can I focus while studying?", "Can you recommend a book?",
           "Explain the water cycle.", "Write a birthday message for my friend.",
           "What are the benefits of budgeting?", "My computer is slow, what can I do?"],
}
SAMPLE_QUESTIONS = ["Give me three tips for staying focused while studying.", "Sağlıklı beslenme için üç öneri ver.",
                    "Write a Python function is_even(n) that returns True if n is even, otherwise False.",
                    "Write a Python function that returns the second smallest number in a list."]
end_id = special_id(tokenizer, END_TOKEN)


def measure(model, label):
    started = time.perf_counter()
    model = model.to(device).eval()
    sampler = Sampler(model, batch=256, max_length=512)
    result = {"label": label}
    for split, tasks in EVAL_SETS.items():
        counts = pass_counts(sampler, tokenizer, tasks, n=N_SAMPLES, temperature=EVAL_TEMPERATURE, seed=7)
        greedy = pass_counts(sampler, tokenizer, tasks, n=1, temperature=0.0, seed=0)
        result[split] = {"tasks": [t.name for t in tasks], "counts": counts, "greedy": greedy,
                         "greedy_pass@1": float(np.mean(greedy)), **summarize_counts(counts, N_SAMPLES, ks=(1, 8, 64))}
    result["reviews_test"] = {lang: {k: v for k, v in evaluate_reviews(model, tokenizer, *reviews[lang], lang,
                                                                       batch_size=64, device=device).items()
                                     if k != "confusion"} for lang in LANGUAGES}
    result["sft_val"] = loss_evaluate(model, sft_val, windows=128, batch_size=16, device=device)
    result["code_reference_loss"] = code_evaluate(model, tokenizer, EVAL_SETS["hand_written"], mode="chat",
                                                  loss_only=True, log=lambda _: None)[1]["reference_loss"]
    labels = label_token_ids(tokenizer)
    result["digit_mass"] = {lang: float(score(model, [prompt_ids(tokenizer, [{"role": "user", "content": q}]) for q in qs],
                                              labels, batch_size=20, device=device)[1].mean())
                            for lang, qs in GENERAL.items()}
    answers = sampler([prompt_ids(tokenizer, [{"role": "user", "content": q}]) for q in SAMPLE_QUESTIONS],
                      max_new_tokens=160, temperature=0.0, seed=0, stop=lambda out: out[-1] == end_id)
    result["samples"] = [{"user": q, "assistant": tokenizer.decode([t for t in a if t != end_id])}
                         for q, a in zip(SAMPLE_QUESTIONS, answers)]
    del sampler
    torch.cuda.empty_cache()
    result["seconds"] = round(time.perf_counter() - started)
    return result


def show(result):
    print(f"--- {result['label']}  ({result['seconds']} s)")
    for split in EVAL_SETS:
        r = result[split]
        print(f"  {split:13s} pass@1 {r['pass@1']:.3f}  pass@8 {r['pass@8']:.3f}  pass@64 {r['pass@64']:.3f}  "
              f"greedy {r['greedy_pass@1']:.3f}")
    print("  reviews F1  " + "  ".join(f"{lang} {m['macro_f1']:.3f}" for lang, m in result["reviews_test"].items())
          + "  | sft val " + "  ".join(f"{k} {v:.3f}" for k, v in result["sft_val"].items())
          + f"  | code ref {result['code_reference_loss']:.3f}  | digit mass "
          + "  ".join(f"{k} {v:.3f}" for k, v in result["digit_mass"].items()))

## 8. Baselines

In [ ]:
before = {}
for size, (model, _) in base.items():
    before[size] = fetch(f"before-{size}")
    if before[size] is None:
        before[size] = measure(model, f"{STARTS[size]} (before RL)")
        model.cpu()
        torch.cuda.empty_cache()
        publish(f"before-{size}", before[size])
    show(before[size])

## 9. Difficulty filter

RLOO learns only from tasks where the answers to one task differ. Each model writes 32 answers to every training task (a trial with 8 kept only 19 tasks for `s-50m`); the tasks it solves at least once but not always form its training set.

In [ ]:
from glm_ladder.rl import RLConfig, RLTrainer

FILTER_N = 32
by_name = {task.name: task for task in TASKS["train"]}
filtered = fetch("train-sets")
if filtered is None:
    filtered = {}
    for size, (model, _) in base.items():
        model.to(device).eval()
        sampler = Sampler(model, batch=256, max_length=512)
        counts = pass_counts(sampler, tokenizer, TASKS["train"], n=FILTER_N, temperature=1.0, seed=11)
        del sampler
        model.cpu()
        torch.cuda.empty_cache()
        filtered[size] = {"counts": dict(zip(by_name, counts)),
                          "kept": [name for name, c in zip(by_name, counts) if 0 < c < FILTER_N]}
    publish("train-sets", filtered)
train_sets = {size: [by_name[name] for name in info["kept"]] for size, info in filtered.items()}
for size, info in filtered.items():
    counts = list(info["counts"].values())
    print(f"{size}: never solved {sum(c == 0 for c in counts)}, always solved {sum(c == FILTER_N for c in counts)}, "
          f"kept {len(info['kept'])} of {len(counts)}")

## 10. The three RL runs

200 steps each, 16 tasks x 16 answers per step, LR 5e-6 (from a 20-step trial). `rl-s50m-001` and `rl-m100m-001` (KL 0.02) are the final models; `rl-s50m-nokl-001` is the ablation without the KL penalty.

Every 25 steps the full training state is pushed to `runs/<run>/`. A run whose final measurement is already in `rl/` is skipped; a run with checkpoints continues from the last one; the reference model for the KL penalty is always the original starting model.

In [ ]:
RUNS = {
    "rl-s50m-001": ("s50m", RLConfig(run_name="rl-s50m-001", kl_coef=0.02, log_every=10)),
    "rl-m100m-001": ("m100m", RLConfig(run_name="rl-m100m-001", kl_coef=0.02, log_every=10)),
    "rl-s50m-nokl-001": ("s50m", RLConfig(run_name="rl-s50m-nokl-001", kl_coef=0.0, log_every=10)),
}
WORKDIR = Path("/content/rl-runs")
after = {}
for run, (size, cfg) in RUNS.items():
    after[run] = fetch(f"after-{run}")
    if after[run] is not None:
        print(f"{run}: finished earlier")
        show(after[run])
        continue
    start_model, start_state = base[size]
    model = copy.deepcopy(start_model).to(device)
    reference = copy.deepcopy(start_model).to(device) if cfg.kl_coef > 0 else None
    trainer = RLTrainer(cfg, model, tokenizer, train_sets[size], device=device, log=print, reference=reference)
    store = RunStore(api, RUNS_REPO, run)
    latest = store.latest()
    if latest is not None:
        pulled = store.pull(latest, WORKDIR / "pulled")
        trainer.restore(pulled)
        print(f"{run}: resumed from {latest}, {len(trainer.history)} of {cfg.steps} steps done")
    else:
        print(f"=== {run}: {len(train_sets[size])} training tasks")
    metadata = {"stage": "rl", "commit": commit, "model_config": start_state["model_config"],
                "init_from": f"{RUNS_REPO}/runs/{STARTS[size]}/{start_state['checkpoint']}",
                "train_tasks": [t.name for t in train_sets[size]]}

    def checkpoint(step, run=run, trainer=trainer, store=store, metadata=metadata, cfg=cfg):
        directory = WORKDIR / run / f"step-{step:06d}"
        tokens = sum(r["answer_tokens"] * cfg.prompts_per_step * cfg.group for r in trainer.history)
        trainer.save(directory, extra={**metadata, "tokens_seen": int(tokens)})
        store.push(directory, milestone=step == cfg.steps)
        shutil.rmtree(directory)                      # on the Hub now; keep the runtime's disk free
        print(f"checkpoint {directory.name} pushed")

    trainer.run(on_checkpoint=checkpoint, checkpoint_every=25)
    del trainer, reference
    torch.cuda.empty_cache()
    after[run] = measure(model, f"{run} (after RL)")
    publish(f"after-{run}", after[run])
    show(after[run])
    model.cpu()
    del model
    torch.cuda.empty_cache()

## 11. Before vs after

Per-task pass rates (64 samples) before and after, paired by task: mean difference and a bootstrap 95% interval. An interval that excludes 0 is a change the sampling noise cannot explain.

In [ ]:
from glm_ladder.rl import paired_bootstrap

comparison = {}
for run, result in after.items():
    size = RUNS[run][0]
    comparison[run] = {}
    print(f"=== {run} vs {STARTS[size]}")
    for split in EVAL_SETS:
        b = np.array(before[size][split]["counts"]) / N_SAMPLES
        a = np.array(result[split]["counts"]) / N_SAMPLES
        stats = paired_bootstrap(b, a)
        comparison[run][split] = stats
        print(f"  {split:13s} pass@1 {b.mean():.3f} -> {a.mean():.3f}   diff {stats['mean']:+.3f} "
              f"[{stats['low']:+.3f}, {stats['high']:+.3f}]  greedy {before[size][split]['greedy_pass@1']:.3f} -> "
              f"{result[split]['greedy_pass@1']:.3f}")
    for lang in LANGUAGES:
        print(f"  reviews {lang} F1 {before[size]['reviews_test'][lang]['macro_f1']:.3f} -> "
              f"{result['reviews_test'][lang]['macro_f1']:.3f}")
    print("  sft val " + "  ".join(f"{k} {before[size]['sft_val'][k]:.3f} -> {result['sft_val'][k]:.3f}" for k in result["sft_val"])
          + f"  | code ref {before[size]['code_reference_loss']:.3f} -> {result['code_reference_loss']:.3f}")
    print("  digit mass " + "  ".join(f"{k} {before[size]['digit_mass'][k]:.3f} -> {result['digit_mass'][k]:.3f}"
                                      for k in result["digit_mass"]))
    for sample in result["samples"]:
        print(f"  USER: {sample['user']}\n  ASSISTANT: {sample['assistant'][:400]}\n")
publish("comparison", {"commit": commit, "gpu": GPU, "n_samples": N_SAMPLES, "eval_temperature": EVAL_TEMPERATURE,
                       "configs": {run: cfg.__dict__ for run, (_, cfg) in RUNS.items()}, "comparison": comparison})

## 12. Done

Close the runtime. Locally:

```bash
python scripts/fetch_evals.py --pattern "rl/*.json" --to artifacts/rl
```